In [6]:
from __future__ import annotations

import argparse
import json
import math
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import MultipleLocator

from utils.utils_read import _sanitize_answer, load_results


DEFAULT_BASE_PATH = Path(
    # "/Users/sebastiancavada/Desktop/tmp_paris/tiny_vqa_creation/output"
    "/data0/sebastian.cavada/compositional-physics/tiny_vqa_creation/output"
)

RUN = "28"

In [11]:
RUN = "28"

DEFAULT_RUNS = [
    f"run_{RUN}_counterfactual_shift",
    f"run_{RUN}_counterfactual_smaller",
    f"run_{RUN}_counterfactual_gravity",
    f"run_{RUN}_general" # this is the baseline
]

RUN_MAP = {
    f"run_{RUN}_general": "Baseline", # this is the baseline
    f"run_{RUN}_counterfactual_shift": "Shift",
    f"run_{RUN}_counterfactual_smaller": "Scaled",
    f"run_{RUN}_counterfactual_gravity": "Gravity",
}
RUN_INDEX = {run_name: idx for idx, run_name in enumerate(RUN_MAP.keys())}

FAMILY_STYLE = {
    "InternVLChat2": {"label": "InternVLChat2", "marker": "o", "color": "#3D73A9"},
    "LLaVAInterleave": {"label": "LLaVAInterleave", "marker": "<", "color": "#4E973F"},
    "LLaVAVideo": {"label": "LLaVAVideo", "marker": "s", "color": "#AAB8CF"},
    "Mantis": {"label": "Mantis", "marker": "^", "color": "#E3873A"},
    "Owl3": {"label": "Owl3", "marker": ">", "color": "#9DCB8C"},
    "Phi": {"label": "Phi", "marker": "v", "color": "#E7C79D"},
    "VILAModel": {"label": "VILAModel", "marker": "p", "color": "#C84039"},
}

FAMILY_ORDER = [
    FAMILY_STYLE["InternVLChat2"]["label"],
    FAMILY_STYLE["LLaVAInterleave"]["label"],
    FAMILY_STYLE["LLaVAVideo"]["label"],
    FAMILY_STYLE["Mantis"]["label"],
    FAMILY_STYLE["Owl3"]["label"],
    FAMILY_STYLE["Phi"]["label"],
    FAMILY_STYLE["VILAModel"]["label"],
]

FAMILY_MARKERS = {
    style["label"]: style["marker"] for style in FAMILY_STYLE.values()
}
FAMILY_COLORS = {
    style["label"]: style["color"] for style in FAMILY_STYLE.values()
}

PLOT_BG = "#FFFFFF"
GRID_MAJOR = "#E6E6E6"
GRID_MINOR = "#F2F2F2"

OBJECT_COUNT_PATTERN = re.compile(r"(?:^|[\\/_-])no-(\d+)(?:$|[\\/_-])")

In [12]:
def _extract_object_count(value: object) -> float:
    if value is None:
        return float("nan")
    if isinstance(value, (list, tuple, set, np.ndarray)):
        text = " ".join(str(v) for v in value)
    else:
        try:
            if pd.isna(value):
                return float("nan")
        except (TypeError, ValueError):
            pass
        text = str(value)

    match = OBJECT_COUNT_PATTERN.search(text)
    if not match:
        return float("nan")
    try:
        return float(int(match.group(1)))
    except (TypeError, ValueError):
        return float("nan")

def _ensure_object_count_column(df: pd.DataFrame) -> pd.DataFrame:
    if "object_count" in df.columns:
        df["object_count"] = pd.to_numeric(df["object_count"], errors="coerce")
        return df

    if "num_objects" in df.columns:
        df["object_count"] = pd.to_numeric(df["num_objects"], errors="coerce")
        return df

    source_cols = [col for col in ["simulation_id", "file_name", "idx"] if col in df.columns]
    if not source_cols:
        return df

    inferred = pd.Series(np.nan, index=df.index, dtype="float64")
    for col in source_cols:
        inferred = inferred.fillna(df[col].apply(_extract_object_count))

    if inferred.notna().any():
        df["object_count"] = inferred
        print("Inferred object_count from cached fields.")

    return df


def load_metadata_map(metadata_path: Path) -> dict[str, dict]:
    with metadata_path.open("r", encoding="utf-8") as f:
        metadata = json.load(f)
    return {str(item["id"]): item for item in metadata if "id" in item}

def build_eval_df(base_path: Path, run_name: str) -> pd.DataFrame:
    results_dir = base_path / run_name / f"results_{run_name}"
    if not results_dir.exists():
        raise FileNotFoundError(
            f"Missing results directory for {run_name}: {results_dir}"
        )

    model_cols = sorted(
        p.stem.replace("_val", "") for p in results_dir.glob("*_val.json")
    )
    if not model_cols:
        raise FileNotFoundError(
            f"No model result files found in directory: {results_dir}"
        )

    try:
        df = load_results(
            base_path,
            run_name,
            merge_model_answers=True,
            model_answers_wide=True,
            model_results_dir=results_dir,
            cache=True,
            add_sim_metadata=True,
        )
    except FileNotFoundError as exc:
        print(
            f"Metadata load failed for {run_name} ({exc}). "
            "Retrying without simulation metadata."
        )
        df = load_results(
            base_path,
            run_name,
            merge_model_answers=True,
            model_answers_wide=True,
            model_results_dir=results_dir,
            cache=False,
            add_sim_metadata=False,
        )
    df = _ensure_object_count_column(df)

    model_cols = [col for col in model_cols if col in df.columns]
    if not model_cols:
        raise ValueError(
            f"No matching model columns after loading results for {run_name}."
        )

    df["answer"] = df["answer"].apply(
        lambda answer: _sanitize_answer(answer, max_prefix_chars=None)
    )

    id_cols = [
        col
        for col in [
            "idx",
            "question_id",
            "category",
            "sub_category",
            "num_objects",
            "object_count",
            "answer",
            "mode_test",
            "mode_val",
            "mode",
        ]
        if col in df.columns
    ]

    eval_df = df.melt(
        id_vars=id_cols,
        value_vars=model_cols,
        var_name="model_id",
        value_name="model_answer",
    )

    valid = eval_df["model_answer"].notna() & eval_df["answer"].notna()
    eval_df["is_correct"] = pd.NA
    eval_df.loc[valid, "is_correct"] = (
        eval_df.loc[valid, "model_answer"] == eval_df.loc[valid, "answer"]
    )

    return eval_df

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description=(
            "Run ablation analysis from test_ablation copy.ipynb as a regular Python "
            "script with formatted family labels and scatter plots."
        )
    )
    parser.add_argument("--base-path", type=Path, default=DEFAULT_BASE_PATH)
    parser.add_argument("--metadata-path", type=Path, default=Path("utils/metadata.json"))
    parser.add_argument("--runs", nargs="*", default=DEFAULT_RUNS)
    parser.add_argument("--min-object-count", type=int, default=5)
    parser.add_argument("--output-dir", type=Path, default=Path("output/test_ablation_copy"))

    args, unknown = parser.parse_known_args()
    return args


def collect_runs(base_path: Path, runs: list[str]) -> pd.DataFrame:
    frames: list[pd.DataFrame] = []
    for run_name in runs:
        print(f"Processing run: {run_name}")
        run_df = build_eval_df(base_path, run_name)
        if run_df.empty:
            raise ValueError(f"Run has no rows after loading: {run_name}")
        run_df["run_name"] = run_name
        frames.append(run_df)

    if not frames:
        raise ValueError("No data available for the selected runs.")

    return pd.concat(frames, ignore_index=True)

In [13]:
args = parse_args()
metadata_map = load_metadata_map(args.metadata_path)
full_df = collect_runs(args.base_path, args.runs)

Processing run: run_28_counterfactual_shift
Processing columns...
Reading scenes simulation...


  0%|          | 0/1280 [00:00<?, ?it/s]

Metadata load failed for run_28_counterfactual_shift ([Errno 2] No such file or directory: '/scratch/project/eu-25-92/composite_physics/dataset/simulation_v4/dl3dv-counterfact/jitter-xy/random/1/c-1_no-1_cl-output-sims-v4-dl3dv-random-1-all_seed-5_all_d-10_s-dl3dv-all_models-hf-gso_MLP-10_smooth_h-10-40_seed-5_20260119_084256/simulation_kinematics_min.json'). Retrying without simulation metadata.
Processing columns...
Merging model answers...


  0%|          | 1/1280 [00:00<04:35,  4.64it/s]


Inferred object_count from cached fields.
Processing run: run_28_counterfactual_smaller
Processing columns...
Reading scenes simulation...


...

  1%|          | 1/110 [00:00<00:05, 20.87it/s]


Metadata load failed for run_28_counterfactual_gravity ([Errno 2] No such file or directory: '/scratch/project/eu-25-92/composite_physics/dataset/simulation_v4/dl3dv-counterfact/low-gravity/random/1/c-1_no-1_cl-output-sims-v4-dl3dv-random-1-all_seed-0_all_d-10_s-dl3dv-all_models-hf-gso_MLP-10_smooth_h-10-40_seed-0_20260119_002159/simulation_kinematics_min.json'). Retrying without simulation metadata.
Processing columns...
Merging model answers...
Inferred object_count from cached fields.
Processing run: run_28_general
Processing columns...
Reading scenes simulation...


 98%|█████████▊| 9792/10000 [00:26<00:00, 3605.88it/s]

Merging model answers...


100%|██████████| 10000/10000 [00:43<00:00, 230.43it/s]


In [15]:
res = full_df[full_df['run_name'] == 'run_28_general']
print(len(res['model_id'].unique()))

54


In [ ]:
res_general = full_df[full_df['run_name'] == 'run_28_general']
res_c_shift = full_df[full_df['run_name'] == 'run_28_counterfactual_shift']